# 05 - BERT Evaluation

Loads the saved BERT artifact from `04_bert_fine_tuning.ipynb` and evaluates it **exactly
once** on the frozen, untouched `test.csv`. Saves document-level predictions and
probabilities, plus accuracy, macro/weighted F1, per-class metrics, confusion matrix, and
inference latency.

This is the first and only notebook that opens `test.csv` for BERT. Calls reusable logic
from `src/newstart_ai/models/bert/` and `src/newstart_ai/evaluation/`.

### Load the frozen split and the evaluation tools

**Purpose:** Load all three splits (train, validation, test) and the shared evaluation
helpers this notebook needs to score BERT's predictions.

**Why this step is necessary:** This is the first and only notebook in the whole project
that is allowed to read `test_df` for BERT. Loading it here, explicitly and just once, is
what keeps the "touch the test set exactly one time" rule enforceable -- every earlier BERT
notebook (03, 04) deliberately never referenced `test_df` at all.

**Inputs:** `data/splits/{train,validation,test}.csv` and `split_manifest.json`.

**Output:** `train_df`, `val_df`, `test_df`, `manifest`, and `settings`.

**How to interpret the result:** The printed row count for the test set (151) should match
notebook 03's saved split exactly, and the accompanying comment is a reminder that this is a
one-time event for BERT, not something that repeats.

In [1]:
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path("..") / "src"))

from newstart_ai.config import load_settings
from newstart_ai.data import load_split
from newstart_ai.models.bert import latest_ready_artifact_id, load_artifact, build_long_document_strategy, BERTClassifier
from newstart_ai.evaluation import evaluate_predictions, save_predictions, save_metrics_report
from newstart_ai.schemas import ClassificationResult

settings = load_settings()
# load_split() gives us all three splits, but this notebook is the ONE place in the BERT
# workflow where test_df is deliberately used -- every earlier BERT notebook avoided it.
train_df, val_df, test_df, manifest = load_split(settings)
print(f"test set: {len(test_df)} rows (touched here for the first and only time)")

test set: 151 rows (touched here for the first and only time)


## Load the READY BERT artifact

Never hard-codes an artifact_id -- picks up whichever artifact 04_bert_fine_tuning most recently marked READY.

### Load the winning BERT artifact by ID, not by guesswork

**Purpose:** Find and load whichever BERT artifact notebook 04 most recently marked
`"ready"`, along with its saved metadata (base model, long-document strategy, and its
validation macro F1).

**Why this step is necessary:** `latest_ready_artifact_id()` looks this up automatically
instead of the notebook hard-coding a specific artifact ID -- so re-running notebook 04 later
(e.g. after a dataset update) and then re-running this notebook would automatically pick up
the newer model, with no manual bookkeeping required. The assertion is a safety check: if no
artifact has ever completed training successfully, this notebook stops immediately with a
clear error rather than failing confusingly further down.

**Inputs:** The `artifacts/models/` directory on disk (specifically, each artifact's
`metadata.json`).

**Output:** `artifact_id` (a string), `model` and `tokenizer` (loaded from disk), and
`metadata` (the same `BertArtifactMetadata` object notebook 04 saved).

**How to interpret the result:** The printed `long_document_strategy` and
`best_validation_macro_f1` should match exactly what notebook 04 reported for its winning
strategy -- this is confirmation that the correct model was loaded.

In [2]:
artifact_id = latest_ready_artifact_id(settings)
assert artifact_id is not None, "No READY BERT artifact found -- run 04_bert_fine_tuning.ipynb first."

model, tokenizer, metadata = load_artifact(settings, artifact_id)
print(f"Loaded artifact {artifact_id}")
print(f"  base_model: {metadata.base_model}")
print(f"  long_document_strategy: {metadata.long_document_strategy}")
print(f"  best_validation_macro_f1: {metadata.validation_metrics.get('best_validation_macro_f1'):.4f}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loaded artifact 3628681550d7433b94407f684946bb2f
  base_model: bert-base-uncased
  long_document_strategy: first_512
  best_validation_macro_f1: 1.0000


### Wire the loaded weights into a ready-to-use classifier

**Purpose:** Build a `BERTClassifier` configured with the *same* long-document strategy the
winning model was trained with, then swap in the actual trained weights and tokenizer that
were just loaded from disk.

**Why this step is necessary:** `BERTClassifier`'s constructor normally downloads a fresh,
untrained base checkpoint -- here we immediately overwrite that with the specific trained
model and tokenizer from notebook 04, so predictions below reflect the fine-tuned model, not
a blank one. Reusing `build_long_document_strategy()` with the artifact's own recorded
strategy (rather than whatever happens to be the current config default) guarantees
documents are chunked at evaluation time exactly the way they were during training.

**Inputs:** `settings`, `metadata.long_document_strategy`, and the `model`/`tokenizer`
loaded in the previous cell.

**Output:** `classifier`, a `BERTClassifier` instance ready to predict on real text.

**How to interpret the result:** No output is printed here -- success just means the next
cell's predictions come from the actual fine-tuned model.

In [3]:
classifier = BERTClassifier(
    settings,
    long_document_strategy=build_long_document_strategy(settings, override=metadata.long_document_strategy),
)
# Replace the freshly-constructed (untrained) model/tokenizer with the specific trained
# weights loaded from the artifact -- this is what makes `classifier` the actual winning
# model from notebook 04, not a new blank one.
classifier.model = model.to(classifier.device)
classifier.tokenizer = tokenizer

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Predict on the frozen test set

Per-document latency is measured individually so mean latency reflects real single-document inference time, matching how the Random Form Routing Demo calls this same classifier.

### Predict on the frozen test set, one document at a time

**Purpose:** Run the loaded classifier over every document in `test_df`, measuring
per-document latency individually and recording each prediction as a `ClassificationResult`
-- the same shared result schema used by the LLM and LLM+RAG evaluation notebooks, so all
three methods' outputs can later be compared apples-to-apples.

**Why this step is necessary:** This is the actual, single, frozen test evaluation for
BERT -- the number this project reports as BERT's real-world performance. Predicting one
document at a time (rather than one large batch) measures latency the same way the future
Random Form Routing Demo would call this classifier: one real document at a time, not a
batch-optimized benchmark number.

**Inputs:** `test_texts`, `test_ids`, `test_labels` (from `test_df`) and `classifier`.

**Output:** `results`, a list of `ClassificationResult` objects -- one per test document --
each holding the predicted label, the full softmax probability vector, latency, and the
ground-truth label for later scoring.

**How to interpret the result:** The final printed count should equal the test-set size
(151). No accuracy number appears yet -- that's computed from `results` a few cells below,
after the predictions are saved.

In [4]:
ds_cfg = settings.base.dataset
test_texts = test_df[ds_cfg.text_column].tolist()
test_ids = test_df[ds_cfg.id_column].astype(str).tolist()
test_labels = test_df[ds_cfg.label_column].tolist()

results = []
for doc_id, text, true_label in zip(test_ids, test_texts, test_labels):
    # Time each prediction individually (rather than timing the whole loop) so latency
    # reflects a single real-time inference call, matching how the future demo app would
    # use this same classifier on one document at a time.
    start = time.perf_counter()
    probs = classifier.predict_proba([text])[0]
    latency_ms = (time.perf_counter() - start) * 1000

    # probs is BERT's softmax output -- the model's own confidence distribution over the
    # four labels. This is a fundamentally different kind of "confidence" from the LLM's
    # self-reported answer or the RAG retriever's similarity score, and it is never mixed
    # with those other signals.
    predicted_label = settings.base.labels[int(probs.argmax())]
    results.append(
        ClassificationResult(
            method="bert",
            document_id=doc_id,
            predicted_label=predicted_label,
            true_label=true_label,
            probabilities=dict(zip(settings.base.labels, [float(p) for p in probs])),
            latency_ms=latency_ms,
            metadata={
                "artifact_id": artifact_id,
                "long_document_strategy": metadata.long_document_strategy,
                "split": "test",
            },
        )
    )

print(f"Predicted {len(results)} test documents.")

Predicted 151 test documents.


## Save row-level predictions

Saved before any summary table is built, per NewStart_AI_MVP.md Section 11.

### Save row-level predictions before computing any summary metric

**Purpose:** Write every individual document's prediction to
`artifacts/predictions/bert_test.json`.

**Why this step is necessary:** Saving the raw, per-document results *before* aggregating
them into accuracy/F1 numbers means the underlying evidence for every metric is always
available for later inspection -- this is exactly what makes the error analysis in notebook
09 possible (it re-reads this same file rather than needing to re-run BERT).

**Inputs:** `results` (the list of `ClassificationResult` objects from the previous cell).

**Output:** A JSON file on disk; no new Python object.

**How to interpret the result:** No output is printed, but the file's existence is what
notebook 09 depends on later.

In [5]:
save_predictions(results, method="bert", split="test", settings=settings)

WindowsPath('D:/USD/Projects/a590/newstart-ai/newstart_ai_benchmark/artifacts/predictions/bert_test.json')

## Compute and save metrics

### Compute and save the test-set metrics report

**Purpose:** Turn the 151 individual predictions into the full set of summary metrics this
project tracks for every method: accuracy, macro/weighted F1, per-class precision/recall/F1,
and a confusion matrix.

**Why this step is necessary:** Macro F1 (not plain accuracy) is this project's primary
metric, because accuracy alone can look deceptively good even if the model always gets the
rare IRS class wrong -- macro F1 weights every class equally regardless of how many test
documents it has. Attaching the IRS small-sample note directly to the saved report means
anyone reading these numbers later (including notebook 09's automated comparison) sees the
caveat alongside the number, not as a fact they have to already know.

**Inputs:** The true and predicted labels from `results`, plus their latencies.

**Output:** `report`, a `MetricsReport` object, immediately saved to
`artifacts/reports/bert_test_metrics.json`.

**How to interpret the result:** `report.macro_f1` is the headline number for BERT in this
project's comparisons. The per-class breakdown (visible in `report.model_dump()`'s output)
is where IRS's small test-set size becomes visible as unusually large swings in precision or
recall from just one or two documents.

In [6]:
report = evaluate_predictions(
    true_labels=[r.true_label for r in results],
    predicted_labels=[r.predicted_label for r in results],
    label_order=settings.base.labels,
    method="bert",
    split="test",
    latencies_ms=[r.latency_ms for r in results],
    notes=[
        "IRS test slice is very small (~4-5 documents); IRS per-class metrics are "
        "statistically noisy and should be reported as uncertain, not precise."
    ],
)
save_metrics_report(report, settings)
report.model_dump()

{'method': 'bert',
 'split': 'test',
 'accuracy': 0.9933774834437086,
 'macro_precision': 0.9583333333333334,
 'macro_recall': 0.99375,
 'macro_f1': 0.974108170310702,
 'weighted_f1': 0.9936365922617914,
 'per_class': [{'label': 'USCIS',
   'precision': 1.0,
   'recall': 1.0,
   'f1': 1.0,
   'support': 51},
  {'label': 'DMV', 'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'support': 55},
  {'label': 'SSA',
   'precision': 1.0,
   'recall': 0.975,
   'f1': 0.9873417721518988,
   'support': 40},
  {'label': 'IRS',
   'precision': 0.8333333333333334,
   'recall': 1.0,
   'f1': 0.9090909090909091,
   'support': 5}],
 'confusion_matrix': [[51, 0, 0, 0],
  [0, 55, 0, 0],
  [0, 0, 39, 1],
  [0, 0, 0, 5]],
 'confusion_matrix_labels': ['USCIS', 'DMV', 'SSA', 'IRS'],
 'mean_latency_ms': 16.419129791540026,
 'total_token_usage': None,
 'total_estimated_cost': None,
 'cost_per_document': None,
 'notes': ['IRS test slice is very small (~4-5 documents); IRS per-class metrics are statistically noisy an

## Update the artifact with test metrics

The artifact's `test_metrics` field is filled in now that the single frozen test evaluation has completed -- this notebook never re-runs or overwrites it afterward.

### Record the test metrics back onto the artifact

**Purpose:** Update the same BERT artifact saved in notebook 04 so its metadata now
includes the frozen test-set results, not just the validation results it was saved with
originally.

**Why this step is necessary:** This keeps the artifact's metadata a complete, self-describing
record of the model's entire history -- training configuration, validation performance, and
now final test performance -- all in one place (`artifacts/models/<artifact_id>/metadata.json`),
rather than scattered across separate files with no link back to the model that produced
them.

**Inputs:** `metadata` (loaded earlier in this notebook) and `report` (just computed).

**Output:** The artifact's `metadata.json` file is overwritten with the updated metadata;
the model weights themselves are unchanged.

**How to interpret the result:** After this cell, the artifact on disk is "complete" -- it
carries both how well the model did during development (validation) and its one honest,
final test result.

In [7]:
from newstart_ai.models.bert import save_artifact

metadata.test_metrics = report.model_dump()
save_artifact(classifier.model, classifier.tokenizer, metadata, settings)
print("Artifact updated with test metrics.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Artifact updated with test metrics.


## Summary

- BERT's frozen test-set macro F1: see `report.macro_f1` above.
- Per-class metrics and the confusion matrix are saved to `artifacts/reports/bert_test_metrics.json`.
- Row-level predictions are saved to `artifacts/predictions/bert_test.json` for
  `09_model_comparison_and_error_analysis.ipynb`.
- Next: `06_llm_evaluation.ipynb` evaluates the Gemini-backed LLM classifier on the same
  `test.csv`.